# 07 · Scaling coupled policy maps

## Question

How does policy synthesis grow as we increase the number of regimes, policies and couplings?

## Concept

[Notebook 06](06_coupled_policy_map.ipynb) maximizes the sum of local scores $J_{i,\Pi(i)}$ minus $\beta$ for each edge whose endpoints use different policies.

**Informatics view.** Exact enumeration visits the $N_\psi^{N_\rho}$ valid policy maps. One-hot encoding introduces $N_q=N_\rho N_\psi$ bits and $2^{N_q}$ possible strings. Only one bit per regime may be set, so the feasible fraction is $(N_\psi/2^{N_\psi})^{N_\rho}$. Uniform random bits give that fraction as their probability of producing a valid map.

**Why it matters here.** More regimes multiply the valid search space. More policies multiply choices in every regime and increase the number of bit pairs per constraint. Denser graphs increase the work per map and add interaction terms without adding bits. A larger positive $\beta$ changes coefficient values and may change the optimum, while the nonzero interaction count stays the same. Setting $\beta=0$ removes the inter-regime terms. A global shared-resource constraint can introduce dependencies even on a sparse graph. Variable count alone misses these differences.

## Worked example

Use prefixes of one fixed illustrative score table. Its first four rows and two columns reproduce Notebook 06. The first sweep changes regime count with two policies. The second adds a third policy at four regimes. The last two sweeps hold the 4×2 score matrix fixed while changing graph density or switching weight.

## Exercise / prediction

Predict which has more valid maps, 4×3 or 6×2. They use the same number of one-hot bits. Which has more quadratic interactions on a chain? At fixed size, which counts change when the chain becomes complete?

In [1]:
from itertools import combinations, product
from math import comb
from statistics import median
from time import perf_counter

import numpy as np
from IPython.display import Markdown, display

local_scores = np.array([
    [4, 0, 2], [1, 4, 3], [4, 0, 2],
    [0, 6, 3], [5, 1, 2], [1, 4, 2],
])
cases = [
    (2, 2, "chain", 2), (3, 2, "chain", 2), (4, 2, "chain", 2),
    (4, 3, "chain", 2), (6, 2, "chain", 2),
    (4, 2, "cycle", 2), (4, 2, "complete", 2),
    (4, 2, "chain", 0), (4, 2, "chain", 1), (4, 2, "chain", 6),
]

def exact_optimum(J, edges, beta):
    n, m = J.shape
    best_score = -np.inf
    winners = []
    for policy_map in product(range(m), repeat=n):
        local = sum(J[i, j] for i, j in enumerate(policy_map))
        switches = sum(policy_map[i] != policy_map[k] for i, k in edges)
        score = local - beta * switches
        if score > best_score:
            best_score, winners = score, [policy_map]
        elif score == best_score:
            winners.append(policy_map)
    return best_score, winners

results = {}
for n, m, graph, beta in cases:
    J = local_scores[:n, :m]
    edges = [(i, i + 1) for i in range(n - 1)]
    if graph == "cycle":
        edges.append((0, n - 1))
    elif graph == "complete":
        edges = list(combinations(range(n), 2))

    # Expand A(sum_j q_ij - 1)^2 and add cross-policy terms on each edge.
    A = int(np.abs(J).sum()) + 1
    linear = -J.ravel() - A
    quadratic = {(i*m + j, i*m + l): 2*A
                 for i in range(n) for j, l in combinations(range(m), 2)}
    if beta:
        quadratic.update({(i*m + j, k*m + l): beta for i, k in edges
                          for j, l in product(range(m), repeat=2) if j != l})
    expected_pairs = n*comb(m, 2) + (len(edges)*m*(m-1) if beta else 0)
    assert len(quadratic) == expected_pairs

    elapsed = []
    for _ in range(7):
        start = perf_counter()
        optimum, winners = exact_optimum(J, edges, beta)
        elapsed.append(perf_counter() - start)
    results[n, m, graph, beta] = dict(
        n=n, m=m, graph=graph, beta=beta, edges=edges, J=J, A=A,
        linear=linear, quadratic=quadratic, optimum=optimum,
        winners=winners, elapsed_us=median(elapsed)*1e6,
    )

phase6 = results[4, 2, "chain", 2]
assert phase6["optimum"] == 13 and phase6["winners"] == [(0, 0, 0, 1)]
assert len(phase6["quadratic"]) == 10

## Result · search space and exact baseline

Each regime adds one one-hot constraint and $\binom{N_\psi}{2}$ within-regime pairs. For nonzero $\beta$, each graph edge contributes $N_\psi(N_\psi-1)$ mismatch pairs. Each nonzero quadratic coefficient produces one $ZZ$ term after $q_i=(1-Z_i)/2$. For these instances each bit also has a nonzero QUBO linear term.

In [2]:
growth = ["| Size | Graph | Edge density | β | Valid maps | Binary strings | Feasible | Qubits | Constraints | Linear | QUBO pairs | ZZ |",
          "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
baseline = ["| Size | Graph | β | Exact map(s) | Best score | Median solve time (µs) |",
            "|---|---|---:|---|---:|---:|"]
for r in results.values():
    n, m, graph, beta = (r[key] for key in ("n", "m", "graph", "beta"))
    valid = m**n
    qubits = n*m
    states = 2**qubits
    linear_count = np.count_nonzero(r["linear"])
    pairs = len(r["quadratic"])
    zz_count = sum(weight / 4 != 0 for weight in r["quadratic"].values())
    density = len(r["edges"]) / comb(n, 2)
    growth.append(f"| {n}×{m} | {graph} | {density:.0%} | {beta} | {valid:,} | {states:,} | "
                  f"{valid/states:.2%} | {qubits} | {n} | {linear_count} | {pairs} | {zz_count} |")
    maps = ", ".join("".join(str(j + 1) for j in p) for p in r["winners"])
    baseline.append(f"| {n}×{m} | {graph} | {beta} | {maps} | {r['optimum']:g} | {r['elapsed_us']:.1f} |")
display(Markdown("\n".join(growth)))
display(Markdown("\n".join(baseline)))

| Size | Graph | Edge density | β | Valid maps | Binary strings | Feasible | Qubits | Constraints | Linear | QUBO pairs | ZZ |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 2×2 | chain | 100% | 2 | 4 | 16 | 25.00% | 4 | 2 | 4 | 4 | 4 |
| 3×2 | chain | 67% | 2 | 8 | 64 | 12.50% | 6 | 3 | 6 | 7 | 7 |
| 4×2 | chain | 50% | 2 | 16 | 256 | 6.25% | 8 | 4 | 8 | 10 | 10 |
| 4×3 | chain | 50% | 2 | 81 | 4,096 | 1.98% | 12 | 4 | 12 | 30 | 30 |
| 6×2 | chain | 33% | 2 | 64 | 4,096 | 1.56% | 12 | 6 | 12 | 16 | 16 |
| 4×2 | cycle | 67% | 2 | 16 | 256 | 6.25% | 8 | 4 | 8 | 12 | 12 |
| 4×2 | complete | 100% | 2 | 16 | 256 | 6.25% | 8 | 4 | 8 | 16 | 16 |
| 4×2 | chain | 50% | 0 | 16 | 256 | 6.25% | 8 | 4 | 8 | 4 | 4 |
| 4×2 | chain | 50% | 1 | 16 | 256 | 6.25% | 8 | 4 | 8 | 10 | 10 |
| 4×2 | chain | 50% | 6 | 16 | 256 | 6.25% | 8 | 4 | 8 | 10 | 10 |

| Size | Graph | β | Exact map(s) | Best score | Median solve time (µs) |
|---|---|---:|---|---:|---:|
| 2×2 | chain | 2 | 12 | 6 | 39.2 |
| 3×2 | chain | 2 | 111 | 9 | 72.3 |
| 4×2 | chain | 2 | 1112 | 13 | 139.7 |
| 4×3 | chain | 2 | 1112 | 13 | 671.0 |
| 6×2 | chain | 2 | 111212, 111222 | 18 | 640.8 |
| 4×2 | cycle | 2 | 1112 | 11 | 181.9 |
| 4×2 | complete | 2 | 1212, 2222 | 10 | 135.3 |
| 4×2 | chain | 0 | 1212 | 18 | 134.4 |
| 4×2 | chain | 1 | 1212 | 15 | 186.5 |
| 4×2 | chain | 6 | 2222 | 10 | 132.6 |

**Exercise check.** The 4×3 chain has 81 valid maps and 30 quadratic terms. The 6×2 chain has 64 valid maps and 16 quadratic terms. Both use 12 qubits and 4,096 binary strings. Turning the 4×2 chain into a complete graph at $\beta=2$ raises its coupling count from 10 to 16 with the same bit count and feasible fraction.

Exact timings are medians of seven local runs of the enumeration loop, including winner tracking. They exclude encoding and notebook rendering. Every displayed case was solved. Enumeration streams the valid maps and keeps only current winners. For this implementation, work grows as $O(N_\psi^{N_\rho}(N_\rho+|E|))$. Larger instances are outside the measured range. A chain admits dynamic programming over the last policy; dense graphs or global constraints need other structured solvers or bounds.

## QAOA scaling preview

Use the two smallest chain cases at $\beta=2$ and the depth-one statevector grid search from Notebook 06. Build the Ising fields and couplings explicitly, then compare the encoded energies to exact feasible scores and check that every ground state is feasible. Simulate the chosen circuit with Aer. Each reported probability uses all shots as the denominator.

A double-precision statevector uses $16\cdot2^{N_q}$ bytes for amplitudes alone, before simulator overhead and temporary arrays. The 12-qubit cases in this study would need 64 KiB just for amplitudes. This preview uses only four and six qubits to isolate the effect of adding one regime at fixed policy count.

In [3]:
import platform

import qiskit
import qiskit_aer
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

backend = AerSimulator()
shots, seed = 8192, 42
gamma_grid = np.linspace(0, 2*np.pi, 32, endpoint=False)
theta_grid = np.linspace(0, np.pi, 24, endpoint=False)

def logical_circuit(fields, zz, gamma, theta):
    circuit = QuantumCircuit(len(fields))
    circuit.h(range(len(fields)))
    for i, field in enumerate(fields):
        if field:
            circuit.rz(2 * gamma * field, i)
    for (i, k), weight in zz.items():
        circuit.rzz(2 * gamma * weight, i, k)
    circuit.rx(2 * theta, range(len(fields)))
    return circuit

preview = []
for key in ((2, 2, "chain", 2), (3, 2, "chain", 2)):
    r = results[key]
    n, m = r["J"].shape
    qubits = n*m
    fields = -r["linear"] / 2
    offset = n*r["A"] + r["linear"].sum() / 2
    zz = {pair: weight / 4 for pair, weight in r["quadratic"].items()}
    for (i, k), weight in zz.items():
        fields[i] -= weight
        fields[k] -= weight
        offset += weight

    # Qiskit basis indices put qubit 0 at the least significant bit.
    bits = (np.arange(2**qubits)[:, None] >> np.arange(qubits)) & 1
    spins = 1 - 2*bits
    energies = offset + spins @ fields
    for (i, k), weight in zz.items():
        energies += weight * spins[:, i] * spins[:, k]
    rows = bits.reshape(-1, n, m)
    feasible = (rows.sum(axis=2) == 1).all(axis=1)
    assignments = rows.argmax(axis=2)
    scores = r["J"][np.arange(n), assignments].sum(axis=1)
    for i, k in r["edges"]:
        scores -= r["beta"] * (assignments[:, i] != assignments[:, k])
    np.testing.assert_array_equal(energies[feasible], -scores[feasible])
    optimal = feasible & (scores == r["optimum"])
    assert np.array_equal(energies == energies.min(), optimal)

    candidates = []
    for gamma, theta in product(gamma_grid, theta_grid):
        circuit = logical_circuit(fields, zz, gamma, theta)
        probabilities = Statevector.from_instruction(circuit).probabilities()
        candidates.append((float(probabilities @ energies), gamma, theta))
    ideal_energy, gamma, theta = min(candidates)
    circuit = logical_circuit(fields, zz, gamma, theta)
    measured = circuit.copy()
    measured.measure_all()
    counts = backend.run(measured, shots=shots, seed_simulator=seed).result().get_counts()
    assert sum(counts.values()) == shots
    sample = np.array([counts.get(f"{index:0{qubits}b}", 0) for index in range(2**qubits)])
    gate_sizes = [item.operation.num_qubits for item in circuit.data]
    preview.append((n, m, qubits, circuit.depth(), gate_sizes.count(1), gate_sizes.count(2),
                    sample[feasible].sum()/shots, sample[optimal].sum()/shots))
    print(f"{n}×{m}: gamma={gamma:.6f}, theta={theta:.6f}, ideal mean energy={ideal_energy:.3f}")

print(f"Python {platform.python_version()}, Qiskit {qiskit.__version__}, Aer {qiskit_aer.__version__}")
print(f"{backend.name}, p=1, grid 32×24, {shots} shots, seed {seed}")
lines = ["| Size | Qubits | Logical depth | 1-qubit gates | 2-qubit gates | Feasible shots | Optimal shots |",
         "|---|---:|---:|---:|---:|---:|---:|"]
for n, m, q, depth, one, two, feasible_rate, optimal_rate in preview:
    lines.append(f"| {n}×{m} | {q} | {depth} | {one} | {two} | {feasible_rate:.2%} | {optimal_rate:.2%} |")
display(Markdown("\n".join(lines)))

2×2: gamma=0.196350, theta=1.178097, ideal mean energy=-2.404


3×2: gamma=0.392699, theta=2.094395, ideal mean energy=2.491
Python 3.15.2, Qiskit 2.6.2, Aer 0.18.4
aer_simulator, p=1, grid 32×24, 8192 shots, seed 42


| Size | Qubits | Logical depth | 1-qubit gates | 2-qubit gates | Feasible shots | Optimal shots |
|---|---:|---:|---:|---:|---:|---:|
| 2×2 | 4 | 5 | 11 | 4 | 87.22% | 28.39% |
| 3×2 | 6 | 6 | 18 | 7 | 57.74% | 5.18% |

### Why scaling matters for real quantum hardware

A quantum processing unit has a limited number of usable physical qubits and a fixed connectivity graph. Its native gate set determines how logical interactions decompose. Connecting distant qubits can require routing operations, which add depth and two-qubit gates. Two-qubit gate errors accumulate, and execution must fit within the device's coherence time, the time over which quantum information remains usable. The logical counts above provide a reference. Later phases will test whether the same problems remain practical after compilation to a real device.

## Takeaway

Valid maps, binary strings, and interactions grow at different rates. The 4×3 and 6×2 cases have the same bit count but different feasible fractions and coupling counts. The two finite-shot previews characterize specific ideal circuits and grids.

## Presentation asset

Use the growth table to contrast 4×3 with 6×2, then show the chain and complete graph at 4×2 to isolate the effect of connectivity.

MkDocs topic for this notebook: `docs/benchmarking.md`.

## Next step

Phase 8, `08_encoding_comparison.ipynb`, will compare encodings on the same exact policy-map instances.